# Fortune conversation logging quality

## tl;dr

Run the aggregate release gate below. It never selects or displays transcript text. A pass confirms structural, privacy, retention, and evaluator-placement checks; it does not authorize production capture.

## Context & Methods

The turn is the primary quality grain. The audit checks identifiers, lifecycle timestamps, transcript shape, privacy boundaries, prompt-context coverage, retention, and evaluator placements.

### Key Assumptions

- `DATABASE_URL` points to the intended Railway environment.
- Prompt policy v2 rows must have stage, request type, and language labels.
- Only complete, privacy-clear synthetic turns may enter evaluation.
- Historical `legacy` rows remain valid but are not counted as v2 context coverage.

In [ ]:
import json
import os
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == 'analysis':
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))
from scripts.audit_conversation_quality import run_audit

## Data

The only input is a database connection supplied through the process environment. Query results are aggregate counts, latency summaries, and bounded categorical distributions.

In [ ]:
audit = run_audit(os.environ.get('DATABASE_URL', ''))
assert audit['quality_gate']['message_content_selected'] is False
audit

## Results

Interpret `quality_gate.status` first. Any listed failure blocks release until corrected and rerun. Review dimension coverage separately because a small or synthetic-only sample may pass structural checks without establishing participant usability.

In [ ]:
summary = {
    'status': audit['quality_gate']['status'],
    'failures': audit['quality_gate']['failures'],
    'conversations': audit['profile']['conversations'],
    'turns': audit['profile']['turns'],
    'p95_ms': audit['latency']['p95_ms'],
}
summary

## Takeaways

Promote only after the gate passes on staging and the live smoke test demonstrates opening, follow-up, privacy hold, English, and Spanish paths. Keep production logging off until Fortune explicitly approves capture, notice, reviewers, and retention.